# Training prep 

In [1]:
# imports for all modules under src/preprocessing/*.py
from importlib import import_module
from pathlib import Path
import sys

# Make project src importable from this notebook location
project_root = Path.cwd().resolve().parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Auto-import every preprocessing module and expose as globals
preprocessing_modules = {}
preprocessing_dir = src_path / "preprocessing"
for module_file in sorted(preprocessing_dir.glob("*.py")):
    if module_file.stem.startswith("_"):
        continue
    module = import_module(f"preprocessing.{module_file.stem}")
    preprocessing_modules[module_file.stem] = module
    globals()[module_file.stem] = module

print("===Loaded preprocessing modules:", ", ".join(preprocessing_modules.keys()))

===Loaded preprocessing modules: Accept, approvalDate, approvalFY, BalanceGross, base_cleaning, city_bank, createJob, DisbursementDate, DisbursementGross, example, franchise_code, LowDoc, newExists, noemp, retainedJob, RevLineCr, urban_rural


In [1]:
# imports for all modules under src/preprocessing/*.py
from importlib import import_module
from pathlib import Path
import sys

# Make project src importable from this notebook location
project_root = Path.cwd().resolve().parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Auto-import every preprocessing module and expose as globals
preprocessing_modules = {}
preprocessing_dir = src_path / "preprocessing"
for module_file in sorted(preprocessing_dir.glob("*.py")):
    if module_file.stem.startswith("_"):
        continue
    module = import_module(f"preprocessing.{module_file.stem}")
    preprocessing_modules[module_file.stem] = module
    globals()[module_file.stem] = module

print("===Loaded preprocessing modules:", ", ".join(preprocessing_modules.keys()))

===Loaded preprocessing modules: Accept, approvalDate, approvalFY, BalanceGross, base_cleaning, city_bank, createJob, DisbursementDate, DisbursementGross, example, franchise_code, LowDoc, newExists, noemp, retainedJob, RevLineCr, urban_rural


In [2]:
import pandas as pd
from pathlib import Path
import importlib

# Load df only if it is not already in memory
if "df" not in globals():
    project_root = Path.cwd().resolve().parent
    df = pd.read_csv(project_root / "data" / "train.csv")

df.head()

,id,LoanNr_ChkDgt,Name,City,State,Bank,BankState,ApprovalDate,ApprovalFY,NoEmp,...,CreateJob,RetainedJob,FranchiseCode,UrbanRural,RevLineCr,LowDoc,DisbursementDate,DisbursementGross,BalanceGross,Accept
0,64afe857c28,9448323000,MIDWEST CRANKSHAFT & ENGINE,HARVEY,IL,JPMORGAN CHASE BANK NATL ASSOC,IL,9-Aug-96,1996,28,...,0,0,1,0,N,N,31-Mar-97,"$600,000.00",$0.00,0
1,1705a7346c2,2854405007,"Iredesign, Limited",CHICAGO,IL,JPMORGAN CHASE BANK NATL ASSOC,IL,10-Dec-07,2008,1,...,1,1,0,1,N,N,31-Dec-07,"$25,400.00",$0.00,1
2,7439801ad8a,9300423010,PHILLY'S INC.,ROCHELLE,IL,BMO HARRIS BK NATL ASSOC,IL,23-May-96,1996,6,...,0,0,1,0,N,Y,30-Sep-96,"$20,000.00",$0.00,1
3,a3f8f9d0611,4349265000,USA Laser Imaging Inc.,Loves park,IL,ALPINE BANK & TRUST CO.,IL,4-Nov-10,2011,5,...,0,5,0,1,N,N,1-Mar-11,"$75,000.00",$0.00,1
4,71e4f243b5d,2433905006,"Dan Morrell, Inc.",LISLE,IL,JPMORGAN CHASE BANK NATL ASSOC,IL,3-May-07,2007,3,...,1,3,0,1,N,N,31-May-07,"$50,000.00",$0.00,0


### Base Cleanup
remove: 'id', 'LoanNr_ChkDgt', 'Name'
### City
cleanup
### Bank
cleanup

### IsLocalBank
created

### State
Removed
### BankState
Removed

In [3]:
base_cleaning = importlib.reload(base_cleaning)
df = base_cleaning.clean_base_columns(df)
df.head()

,City,Bank,ApprovalDate,ApprovalFY,NoEmp,NewExist,CreateJob,RetainedJob,FranchiseCode,UrbanRural,RevLineCr,LowDoc,DisbursementDate,DisbursementGross,BalanceGross,Accept,IsLocalBank
0,HARVEY,JPMORGAN CHASE BANK NATL ASSOC,9-Aug-96,1996,28,1.0,0,0,1,0,N,N,31-Mar-97,"$600,000.00",$0.00,0,1
1,CHICAGO,JPMORGAN CHASE BANK NATL ASSOC,10-Dec-07,2008,1,2.0,1,1,0,1,N,N,31-Dec-07,"$25,400.00",$0.00,1,1
2,ROCHELLE,BMO HARRIS BK NATL ASSOC,23-May-96,1996,6,2.0,0,0,1,0,N,Y,30-Sep-96,"$20,000.00",$0.00,1,1
3,LOVES PARK,ALPINE BANK & TRUST CO.,4-Nov-10,2011,5,2.0,0,5,0,1,N,N,1-Mar-11,"$75,000.00",$0.00,1,1
4,LISLE,JPMORGAN CHASE BANK NATL ASSOC,3-May-07,2007,3,1.0,1,3,0,1,N,N,31-May-07,"$50,000.00",$0.00,0,1


###  NoEmp 
* "raw" -> en caso de árboles.( does nothing) <br>
* "log" -> En caso de KNN o SVM for a  logarithmic transformation of NoEmp using log1p <br>
* "binning" -> (creates bins)Para análisis interpretativo, probar en arboles también para ver si mejora o empeora el rendimiento.

### NoEmp_Log
created with log, removes <b>NoEmp</b>

### NoEmp_Bin
created with binning, removes <b>NoEmp</b>

In [4]:
# noemp_option = "raw"   # o "log" o "binning"
noemp_option = "log"
# noemp_option = "binning"

noemp = importlib.reload(noemp)
df = noemp.preprocess_noemp(df, option=noemp_option)
df.head()

,City,Bank,ApprovalDate,ApprovalFY,NewExist,CreateJob,RetainedJob,FranchiseCode,UrbanRural,RevLineCr,LowDoc,DisbursementDate,DisbursementGross,BalanceGross,Accept,IsLocalBank,NoEmp_Log
0,HARVEY,JPMORGAN CHASE BANK NATL ASSOC,9-Aug-96,1996,1.0,0,0,1,0,N,N,31-Mar-97,"$600,000.00",$0.00,0,1,3.367296
1,CHICAGO,JPMORGAN CHASE BANK NATL ASSOC,10-Dec-07,2008,2.0,1,1,0,1,N,N,31-Dec-07,"$25,400.00",$0.00,1,1,0.693147
2,ROCHELLE,BMO HARRIS BK NATL ASSOC,23-May-96,1996,2.0,0,0,1,0,N,Y,30-Sep-96,"$20,000.00",$0.00,1,1,1.945910
3,LOVES PARK,ALPINE BANK & TRUST CO.,4-Nov-10,2011,2.0,0,5,0,1,N,N,1-Mar-11,"$75,000.00",$0.00,1,1,1.791759
4,LISLE,JPMORGAN CHASE BANK NATL ASSOC,3-May-07,2007,1.0,1,3,0,1,N,N,31-May-07,"$50,000.00",$0.00,0,1,1.386294


### City y Bank
binary encoding <br>
* remove city column
* remove bank column

In [6]:
city_bank = importlib.reload(city_bank)

df = city_bank.get_city_bank_encoder(df)

print("Current df features:")
for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")

df.head()



Current df features:
1. City_0
2. City_1
3. City_2
4. City_3
5. City_4
6. City_5
7. City_6
8. City_7
9. City_8
10. City_9
11. City_10
12. Bank_0
13. Bank_1
14. Bank_2
15. Bank_3
16. Bank_4
17. Bank_5
18. Bank_6
19. Bank_7
20. Bank_8
21. ApprovalDate
22. ApprovalFY
23. NewExist
24. CreateJob
25. RetainedJob
26. FranchiseCode
27. UrbanRural
28. RevLineCr
29. LowDoc
30. DisbursementDate
31. DisbursementGross
32. BalanceGross
33. Accept
34. IsLocalBank
35. NoEmp_Log


,City_0,City_1,City_2,City_3,City_4,City_5,City_6,City_7,City_8,City_9,...,FranchiseCode,UrbanRural,RevLineCr,LowDoc,DisbursementDate,DisbursementGross,BalanceGross,Accept,IsLocalBank,NoEmp_Log
0,0,0,0,0,0,0,0,0,0,0,...,1,0,N,N,31-Mar-97,"$600,000.00",$0.00,0,1,3.367296
1,0,0,0,0,0,0,0,0,0,1,...,0,1,N,N,31-Dec-07,"$25,400.00",$0.00,1,1,0.693147
2,0,0,0,0,0,0,0,0,0,1,...,1,0,N,Y,30-Sep-96,"$20,000.00",$0.00,1,1,1.945910
3,0,0,0,0,0,0,0,0,1,0,...,0,1,N,N,1-Mar-11,"$75,000.00",$0.00,1,1,1.791759
4,0,0,0,0,0,0,0,0,1,0,...,0,1,N,N,31-May-07,"$50,000.00",$0.00,0,1,1.386294


### NewExist
* create is_new_business
* create newexist_missing_or_invalid (option A)
* remove NewExist

In [ ]:
# calling src\preprocessing\newExists.py

# Reload module to pick up latest code changes
newExists = importlib.reload(newExists)

# Choose NewExist preprocessing path: 'A' or 'B'
newexist_option = "A" #===> is_new_business +  newexist_missing_or_invalid
# Option A: Create 'is_new_business' and 'newexist_missing_or_invalid' columns
# Option B: Create only 'is_new_business' column, and delete rows with missing/invalid values

print(f"NewExist option used: {newexist_option}")
print(f"Rows before: {len(df):,}")

# Apply preprocessing from the imported module
df = newExists.preprocess_newexist(
    df=df,
    option=newexist_option,
    source_col="NewExist",
)


print(f"Rows after: {len(df):,}")

# Quick check of the generated columns
cols_to_show = ["NewExist", "is_new_business"]
if "newexist_missing_or_invalid" in df.columns:
    cols_to_show.append("newexist_missing_or_invalid")

display(df[cols_to_show].head(10))

# Show only rows flagged as missing/invalid in Option A
if "newexist_missing_or_invalid" in df.columns:
    display(
        df.loc[
            df["newexist_missing_or_invalid"] == 1,
            ["NewExist", "is_new_business", "newexist_missing_or_invalid"],
        ].head(20)
    )

NewExist option used: A
Rows before: 20,768
Rows after: 20,768


,NewExist,is_new_business,newexist_missing_or_invalid
0,1.0,0,0
1,2.0,1,0
2,2.0,1,0
3,2.0,1,0
4,1.0,0,0
5,2.0,1,0
6,1.0,0,0
7,1.0,0,0
8,1.0,0,0
9,1.0,0,0


,NewExist,is_new_business,newexist_missing_or_invalid
1080,0.0,0,1
2995,0.0,0,1
3226,0.0,0,1
3635,0.0,0,1
4503,0.0,0,1
6016,0.0,0,1
6630,0.0,0,1
10090,0.0,0,1
10473,<NA>,0,1
10730,<NA>,0,1


In [ ]:
# calling src\preprocessing\newExists.py

# Reload module to pick up latest code changes
newExists = importlib.reload(newExists)

# Choose NewExist preprocessing path: 'A' or 'B'
newexist_option = "B"

print(f"NewExist option used: {newexist_option}")
print(f"Rows before: {len(df):,}")

# Apply preprocessing from the imported module
df = newExists.preprocess_newexist(
    df=df,
    option=newexist_option,
    source_col="NewExist",
)


print(f"Rows after: {len(df):,}")

# Quick check of the generated columns
cols_to_show = ["NewExist", "is_new_business"]
if "newexist_missing_or_invalid" in df.columns:
    cols_to_show.append("newexist_missing_or_invalid")

display(df[cols_to_show].head(10))

# Show only rows flagged as missing/invalid in Option A
if "newexist_missing_or_invalid" in df.columns:
    display(
        df.loc[
            df["newexist_missing_or_invalid"] == 1,
            ["NewExist", "is_new_business", "newexist_missing_or_invalid"],
        ].head(20)
    )

NewExist option used: B
Rows before: 20,768
Rows after: 20,765


,NewExist,is_new_business
0,1.0,0
1,2.0,1
2,2.0,1
3,2.0,1
4,1.0,0
5,2.0,1
6,1.0,0
7,1.0,0
8,1.0,0
9,1.0,0


### CreateJob
* adds createjob_normalized  (option A)
* adds createjob_standardized  (option B)
* removes CreateJob column 


In [ ]:
# calling src\preprocessing\createJob.py

# Reload module to pick up latest code changes
createJob = importlib.reload(createJob)

# Choose CreateJob preprocessing path: 'A' (normalize) or 'B' (standardize)
createjob_option = "A"

print(f"CreateJob option used: {createjob_option}")
print(f"Rows before: {len(df):,}")

# Apply preprocessing from the imported module
df = createJob.preprocess_createjob(
    df=df,
    option=createjob_option,
    source_col="CreateJob",
)


print(f"Rows after: {len(df):,}")

# Quick check of generated columns
cols_to_show = ["CreateJob"]
if "createjob_normalized" in df .columns:
    cols_to_show.append("createjob_normalized")
if "createjob_standardized" in df.columns:
    cols_to_show.append("createjob_standardized")

display(df[cols_to_show].head(10))

CreateJob option used: A
Rows before: 20,768
Rows after: 20,768


,CreateJob,createjob_normalized
0,0,0.0
1,1,0.000114
2,0,0.0
3,0,0.0
4,1,0.000114
5,0,0.0
6,1,0.000114
7,0,0.0
8,6,0.000682
9,1,0.000114


In [ ]:
# calling src\preprocessing\createJob.py

# Reload module to pick up latest code changes
createJob = importlib.reload(createJob)

# Choose CreateJob preprocessing path: 'A' (normalize) or 'B' (standardize)
createjob_option = "B"

print(f"CreateJob option used: {createjob_option}")
print(f"Rows before: {len(df):,}")

# Apply preprocessing from the imported module
df = createJob.preprocess_createjob(
    df=df,
    option=createjob_option,
    source_col="CreateJob",
)


print(f"Rows after: {len(df):,}")

# Quick check of generated columns
cols_to_show = ["CreateJob"]
if "createjob_normalized" in df .columns:
    cols_to_show.append("createjob_normalized")
if "createjob_standardized" in df.columns:
    cols_to_show.append("createjob_standardized")

display(df[cols_to_show].head(10))

CreateJob option used: B
Rows before: 20,768
Rows after: 20,768


,CreateJob,createjob_standardized
0,0,-0.03393
1,1,-0.028997
2,0,-0.03393
3,0,-0.03393
4,1,-0.028997
5,0,-0.03393
6,1,-0.028997
7,0,-0.03393
8,6,-0.004332
9,1,-0.028997


In [9]:
createJob = importlib.reload(createJob)

# Choose CreateJob preprocessing path: 'A' (normalize) or 'B' (standardize)
createjob_option = "B"

# Apply preprocessing from the imported module
df = createJob.preprocess_createjob(
    df=df,
    option=createjob_option,
    source_col="CreateJob",
)
df.head()

,City_0,City_1,City_2,City_3,City_4,City_5,City_6,City_7,City_8,City_9,...,LowDoc,DisbursementDate,DisbursementGross,BalanceGross,Accept,IsLocalBank,NoEmp_Log,is_new_business,newexist_missing_or_invalid,createjob_standardized
0,0,0,0,0,0,0,0,0,0,0,...,N,31-Mar-97,"$600,000.00",$0.00,0,1,3.367296,0,0,-0.033931
1,0,0,0,0,0,0,0,0,0,1,...,N,31-Dec-07,"$25,400.00",$0.00,1,1,0.693147,1,0,-0.028998
2,0,0,0,0,0,0,0,0,0,1,...,Y,30-Sep-96,"$20,000.00",$0.00,1,1,1.945910,1,0,-0.033931
3,0,0,0,0,0,0,0,0,1,0,...,N,1-Mar-11,"$75,000.00",$0.00,1,1,1.791759,1,0,-0.033931
4,0,0,0,0,0,0,0,0,1,0,...,N,31-May-07,"$50,000.00",$0.00,0,1,1.386294,0,0,-0.028998


### RetainedJob
* adds retainedjob_normalized  (option A)
* adds retainedjob_standardized  (option B)
* removes RetainedJob column 

In [ ]:
# calling src\preprocessing\retainedJob.py

# Reload module to pick up latest code changes
retainedJob = importlib.reload(retainedJob)

# Choose RetainedJob preprocessing path: 'A' (normalize) or 'B' (standardize)
retainedjob_option = "A"
# retainedjob_option = "B"

print(f"RetainedJob option used: {retainedjob_option}")
print(f"Rows before: {len(df):,}")

# Apply preprocessing from the imported module
df = retainedJob.preprocess_retainedjob(
    df=df,
    option=retainedjob_option,
    source_col="RetainedJob",
)


print(f"Rows after: {len(df):,}")

# Quick check of generated columns
cols_to_show = ["RetainedJob"]
if "retainedjob_normalized" in df.columns:
    cols_to_show.append("retainedjob_normalized")
if "retainedjob_standardized" in df.columns:
    cols_to_show.append("retainedjob_standardized")

display(df[cols_to_show].head(10))

RetainedJob option used: A
Rows before: 20,768
Rows after: 20,768


,RetainedJob,retainedjob_normalized
0,0,0.0
1,1,0.000114
2,0,0.0
3,5,0.000568
4,3,0.000341
5,0,0.0
6,4,0.000455
7,0,0.0
8,0,0.0
9,6,0.000682


In [12]:
# Reload module to pick up latest code changes
retainedJob = importlib.reload(retainedJob)

# Choose RetainedJob preprocessing path: 'A' (normalize) or 'B' (standardize)
retainedjob_option = "A"
# retainedjob_option = "B"

# Apply preprocessing from the imported module
df = retainedJob.preprocess_retainedjob(
    df=df,
    option=retainedjob_option,
    source_col="RetainedJob",
)
df.head()

,City_0,City_1,City_2,City_3,City_4,City_5,City_6,City_7,City_8,City_9,...,DisbursementDate,DisbursementGross,BalanceGross,Accept,IsLocalBank,NoEmp_Log,is_new_business,newexist_missing_or_invalid,createjob_standardized,retainedjob_normalized
0,0,0,0,0,0,0,0,0,0,0,...,31-Mar-97,"$600,000.00",$0.00,0,1,3.367296,0,0,-0.033931,0.0
1,0,0,0,0,0,0,0,0,0,1,...,31-Dec-07,"$25,400.00",$0.00,1,1,0.693147,1,0,-0.028998,0.000114
2,0,0,0,0,0,0,0,0,0,1,...,30-Sep-96,"$20,000.00",$0.00,1,1,1.945910,1,0,-0.033931,0.0
3,0,0,0,0,0,0,0,0,1,0,...,1-Mar-11,"$75,000.00",$0.00,1,1,1.791759,1,0,-0.033931,0.000568
4,0,0,0,0,0,0,0,0,1,0,...,31-May-07,"$50,000.00",$0.00,0,1,1.386294,0,0,-0.028998,0.000341


### ApprovalDate

Option A: Parse to datetime.
* ApprovalYear 
* ApprovalMonth.<br>
Fills missing year/month with each column’s mode. - Note: with the most common year and most common month in the column.
* ApprovalDate column removed.

Option B:
Parses ApprovalDate and keeps it as the original column (now datetime).

Use Option A when your model needs numeric engineered time features.
Use Option B when you want to keep the raw date column for later processing.

In [15]:
import importlib
import sys
from pathlib import Path

# 1. Asegurar que el notebook encuentre la carpeta src (estilo de tu equipo)
project_root = Path.cwd().resolve().parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# 2. Importar tus módulos
from preprocessing import approvalDate


# =======================================================
# ApprovalDate
# =======================================================
approvalDate = importlib.reload(approvalDate)
approvaldate_option = "A"

print(f"\nOpción de ApprovalDate utilizada: {approvaldate_option}")
print(f"Filas antes: {len(df):,}")

df = approvalDate.preprocess_approvaldate(
    df=df, # ATENCIÓN: Usamos el df base para probar independientemente
    option=approvaldate_option,
    source_col="ApprovalDate",
)


print(f"Filas después: {len(df):,}")

if approvaldate_option == "A":
    cols_to_show = ["ApprovalYear", "ApprovalMonth"]
    print("\nOpción A seleccionada: Se crearon 'ApprovalYear' y 'ApprovalMonth'.")
    display(df[cols_to_show].head(10))
elif approvaldate_option == "B":
    cols_to_show = ["ApprovalDate"] 
    display(df[cols_to_show].head(10))


Opción de ApprovalDate utilizada: A
Filas antes: 20,765
Filas después: 20,765

Opción A seleccionada: Se crearon 'ApprovalYear' y 'ApprovalMonth'.


,ApprovalYear,ApprovalMonth
0,1996,8
1,2007,12
2,1996,5
3,2010,11
4,2007,5
5,2003,3
6,2005,10
7,1993,2
8,1999,5
9,2004,3


### ApprovalFY
Option A:
* Removes the ApprovalFY

Option B:
* Keeps ApprovalFY.
Converts it to numeric with invalid values coerced to missing.
Fills missing values with the mode (most frequent fiscal year).
Casts to int.
So: A = drop the feature, B = keep/clean/impute the feature.

In [16]:
import importlib
import sys
from pathlib import Path

# 1. Asegurar que el notebook encuentre la carpeta src (estilo de tu equipo)
project_root = Path.cwd().resolve().parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# 2. Importar tus módulos
from preprocessing import approvalFY

# =======================================================
# ApprovalFY
# =======================================================
approvalFY = importlib.reload(approvalFY)
approvalfy_option = "A"

print(f"Opción de ApprovalFY utilizada: {approvalfy_option}")
print(f"Filas antes: {len(df):,}")

df = approvalFY.preprocess_approvalfy(
    df=df,
    option=approvalfy_option,
    source_col="ApprovalFY",
)

print(f"Filas después: {len(df):,}")

if approvalfy_option == "A":
    print("\nOpción A seleccionada: La columna 'ApprovalFY' fue eliminada.")
    display(df.head(5))
elif approvalfy_option == "B":
    display(df[["ApprovalFY"]].head(10))

Opción de ApprovalFY utilizada: A
Filas antes: 20,765
Filas después: 20,765

Opción A seleccionada: La columna 'ApprovalFY' fue eliminada.


,City_0,City_1,City_2,City_3,City_4,City_5,City_6,City_7,City_8,City_9,...,BalanceGross,Accept,IsLocalBank,NoEmp_Log,is_new_business,newexist_missing_or_invalid,createjob_standardized,retainedjob_normalized,ApprovalYear,ApprovalMonth
0,0,0,0,0,0,0,0,0,0,0,...,$0.00,0,1,3.367296,0,0,-0.033931,0.0,1996,8
1,0,0,0,0,0,0,0,0,0,1,...,$0.00,1,1,0.693147,1,0,-0.028998,0.000114,2007,12
2,0,0,0,0,0,0,0,0,0,1,...,$0.00,1,1,1.945910,1,0,-0.033931,0.0,1996,5
3,0,0,0,0,0,0,0,0,1,0,...,$0.00,1,1,1.791759,1,0,-0.033931,0.000568,2010,11
4,0,0,0,0,0,0,0,0,1,0,...,$0.00,0,1,1.386294,0,0,-0.028998,0.000341,2007,5


In [ ]:
FranchiseCode

In [16]:
# llamando a src\preprocessing\franchise_code.py

# Primero importar el módulo
import src.preprocessing.franchise_code as franchise_code

# Recargar el módulo para aplicar los últimos cambios en el código
import importlib
franchise_code = importlib.reload(franchise_code)

# Elegir la ruta de preprocesamiento de FranchiseCode: 'binary' o 'raw'
franchise_option = "binary" 

# Aplicar el preprocesamiento desde el módulo importado
df_franchise = franchise_code.preprocess_franchise_code(
    df=df,
    option=franchise_option,
    source_col="FranchiseCode",
)

print(f"Opción de FranchiseCode utilizada: {franchise_option}")
print(f"Filas antes: {len(df):,}")
print(f"Filas después: {len(df_franchise):,}")

# Comprobación rápida de las columnas generadas
cols_to_show = []
if "FranchiseCode" in df_franchise.columns:
    cols_to_show.append("FranchiseCode")
if "IsFranchise" in df_franchise.columns:
    cols_to_show.append("IsFranchise")

display(df_franchise[cols_to_show].head(10))


Opción de FranchiseCode utilizada: binary
Filas antes: 20,768
Filas después: 20,768


,IsFranchise
0,0
1,0
2,0
3,0
4,0
5,1
6,0
7,1
8,0
9,0


In [ ]:
UrbanRural

In [17]:
# llamando a src\preprocessing\urban_rural.py

# Primero importar el módulo
import src.preprocessing.urban_rural as urban_rural

# Recargar el módulo para aplicar los últimos cambios en el código
import importlib
urban_rural = importlib.reload(urban_rural)

# Elegir la ruta de preprocesamiento de UrbanRural: 'onehot' o 'text'
urbanrural_option = "onehot" 

# Aplicar el preprocesamiento desde el módulo importado
df_urbanrural = urban_rural.preprocess_urban_rural(
    df=df,
    option=urbanrural_option,
    source_col="UrbanRural",
)

print(f"Opción de UrbanRural utilizada: {urbanrural_option}")
print(f"Filas antes: {len(df):,}")
print(f"Filas después: {len(df_urbanrural):,}")

# Comprobación rápida de las columnas generadas
cols_to_show = []
if "UrbanRural" in df_urbanrural.columns:
    cols_to_show.append("UrbanRural")

# Añadir las columnas one-hot generadas si existen
zone_cols = [col for col in df_urbanrural.columns if col.startswith('Zone_')]
cols_to_show.extend(zone_cols)

display(df_urbanrural[cols_to_show].head(10))

Opción de UrbanRural utilizada: onehot
Filas antes: 20,768
Filas después: 20,768


,Zone_Rural,Zone_Undefined,Zone_Urban
0,0,1,0
1,0,0,1
2,0,1,0
3,0,0,1
4,0,0,1
5,0,0,1
6,0,0,1
7,0,1,0
8,0,1,0
9,0,0,1


# RevLineCr

In [6]:
# calling src\preprocessing\RevLineCr.py
RevLineCr = importlib.reload(RevLineCr)

# Aplicar opción A
df_revlinecr = RevLineCr.preprocess_revlinecr(
    df=df,
    option="A",
    source_col="RevLineCr",
)

print(f"Rows before: {len(df):,}")
print(f"Rows after: {len(df_revlinecr):,}")

# Mostrar columnas relevantes
cols_to_show = ["RevLineCr_clean"]  

rev_cols = [col for col in df_revlinecr.columns if col.startswith("revlinecr_")]
cols_to_show += rev_cols

display(df_revlinecr[cols_to_show].head(10))

Rows before: 20,768
Rows after: 20,768


,RevLineCr_clean,revlinecr_is_nonstandard,revlinecr_is_missing
0,N,0,0
1,N,0,0
2,N,0,0
3,N,0,0
4,N,0,0
5,UNKNOWN,1,0
6,N,0,0
7,N,0,0
8,N,0,0
9,UNKNOWN,1,0


In [8]:
# calling src\preprocessing\RevLineCr.py
RevLineCr = importlib.reload(RevLineCr)

df_revlinecr = RevLineCr.preprocess_revlinecr(
    df=df,
    option="B",
    source_col="RevLineCr",
)

print(f"Rows before: {len(df):,}")
print(f"Rows after: {len(df_revlinecr):,}")

cols_to_show = ["RevLineCr_clean"]

ev_cols = [col for col in df_revlinecr.columns if col.startswith("revlinecr_")]
cols_to_show += rev_cols

display(df_revlinecr[cols_to_show].head(10))

Rows before: 20,768
Rows after: 20,768


,RevLineCr_clean
0,N
1,N
2,N
3,N
4,N
5,UNKNOWN
6,N
7,N
8,N
9,UNKNOWN


# LowDoc

In [3]:
# calling src\preprocessing\LowDoc.py
LowDoc = importlib.reload(LowDoc)

# Aplicar opción A
df_lowdoc = LowDoc.preprocess_lowdoc(
    df=df,
    option="A",
    source_col="LowDoc",
)

print(f"Rows before: {len(df):,}")
print(f"Rows after: {len(df_lowdoc):,}")

# Mostrar columnas relevantes
cols_to_show = ["LowDoc_clean"] 

lowdoc_cols = [col for col in df_lowdoc.columns if col.startswith("lowdoc_")]
cols_to_show += lowdoc_cols

display(df_lowdoc[cols_to_show].head(10))

Rows before: 20,768
Rows after: 20,768


,LowDoc_clean,lowdoc_is_nonstandard,lowdoc_is_missing
0,N,0,0
1,N,0,0
2,Y,0,0
3,N,0,0
4,N,0,0
5,Y,0,0
6,N,0,0
7,N,0,0
8,N,0,0
9,N,0,0


In [5]:
# calling src\preprocessing\LowDoc.py
LowDoc = importlib.reload(LowDoc)

# Aplicar opción B
df_lowdoc = LowDoc.preprocess_lowdoc(
    df=df,
    option="B",
    source_col="LowDoc",
)

print(f"Rows before: {len(df):,}")
print(f"Rows after: {len(df_lowdoc):,}")

# Mostrar columnas relevantes
cols_to_show = ["LowDoc_clean"]  

lowdoc_cols = [col for col in df_lowdoc.columns if col.startswith("lowdoc_")]
cols_to_show += lowdoc_cols

display(df_lowdoc[cols_to_show].head(10))

Rows before: 20,768
Rows after: 20,768


,LowDoc_clean
0,N
1,N
2,Y
3,N
4,N
5,Y
6,N
7,N
8,N
9,N
